# THGS Pipeline
**Training-Free Hierarchical Scene Understanding for Gaussian Splatting with Superpoint Graphs**

이 노트북은 THGS 논문의 전체 파이프라인을 Colab 환경에서 실행합니다.

| 단계 | 내용 |
|------|------|
| Cell 0 | 환경 설정 & 의존성 설치 |
| Cell 1 | 데이터셋 다운로드 & 2DGS 장면 준비 |
| Cell 2 | 언어 피처 생성 (SAM + CLIP) |
| Cell 3 | THGS 파이프라인 (인접 그래프 → 엣지 가중치 → 슈퍼포인트 분할 → 계층적 병합) |
| Cell 4 | 평가 (Open-vocabulary Segmentation) |
| Cell 5 | 시각화 |

---
## Cell 0. 환경 설정 & 의존성 설치

In [ ]:
# GPU 확인
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GitHub 레포에서 THGS 클론
import os

REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"

if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull && git submodule update --init --recursive

os.chdir(WORK_DIR)
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
# 의존성 설치

# 0) NumPy 2.x와 호환되는 opencv 설치를 위해 먼저 업그레이드
!pip install -q --upgrade opencv-python

# 1) 기본 pip 패키지
!pip install -q plyfile==0.8.1 trimesh kiui pymeshlab open3d scipy \
    omegaconf open_clip_torch transformations transformers yapf \
    pycocotools mediapy lpips scikit-image einops dearpygui \
    h5py colorhash seaborn pyrootutils hydra-core hydra-colorlog \
    hydra-submitit-launcher numba pytorch-lightning rich \
    ipyfilechooser natsort gdown ninja

# 2) PyTorch Geometric
import torch
CUDA_VERSION = torch.version.cuda.replace(".", "")
TORCH_VERSION = torch.__version__.split("+")[0]
print(f"PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}")

!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_cluster \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html \
    || pip install -q torch_scatter torch_cluster

In [ ]:
# 3) CUDA 서브모듈 빌드
import torch, os

# CUDA_HOME 설정 & GPU 아키텍처 자동 감지
CUDA_HOME = os.environ.get("CUDA_HOME", "/usr/local/cuda")
os.environ["CUDA_HOME"] = CUDA_HOME
cc = torch.cuda.get_device_capability()
TORCH_CUDA_ARCH = f"{cc[0]}.{cc[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = TORCH_CUDA_ARCH
print(f"CUDA_HOME: {CUDA_HOME}")
print(f"GPU arch: {TORCH_CUDA_ARCH} ({torch.cuda.get_device_name(0)})")

# simple-knn 소스 패치 (CUDA 12.x / Blackwell 호환)
# 1) FLT_MAX 미정의 → #include <cfloat> 추가, 중복 #define __CUDACC__ 제거
!sed -i 's|#include "simple_knn.h"|#include "simple_knn.h"\n#include <cfloat>|' submodules/simple-knn/simple_knn.cu
!sed -i '/#define __CUDACC__/d' submodules/simple-knn/simple_knn.cu
# 2) deprecated data<float>() → data_ptr<float>()
!sed -i 's/\.data<float>()/\.data_ptr<float>()/g' submodules/simple-knn/spatial.cu

# 빌드
!pip install submodules/diff-surfel-rasterization 2>&1 | tail -3
!pip install submodules/simple-knn 2>&1 | tail -3

# 빌드 검증
try:
    from diff_surfel_rasterization import GaussianRasterizer
    print("[OK] diff-surfel-rasterization")
except Exception as e:
    print(f"[FAIL] diff-surfel-rasterization: {e}")

try:
    import simple_knn
    print("[OK] simple-knn")
except Exception as e:
    print(f"[FAIL] simple-knn: {e}")

In [ ]:
# 4) SPT 의존성 빌드
!pip install -q ext/spt/dependencies/FRNN
!pip install -q ext/spt/dependencies/FRNN/external/prefix_sum

# grid_graph + parallel_cut_pursuit C++ 확장 컴파일
!python scripts/setup_dependencies.py build_ext

In [ ]:
# 5) 추가 의존성
!pip install -q git+https://github.com/facebookresearch/pytorch3d.git
!pip install -q git+https://github.com/drprojects/point_geometric_features.git
!pip install -q git+https://github.com/minghanqin/segment-anything-langsplat.git

In [ ]:
# 설치 검증
import torch
import open_clip
import torch_geometric
from diff_surfel_rasterization import GaussianRasterizer

print("-- 설치 검증 완료 --")
print(f"  torch: {torch.__version__}")
print(f"  torch_geometric: {torch_geometric.__version__}")
print(f"  open_clip: {open_clip.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

---
## Cell 1. 데이터셋 다운로드 & 2DGS 장면 준비

In [ ]:
# ============================================================
# 설정: 사용할 데이터셋과 장면 선택
# ============================================================
DATASET = "lerf"                                    # "lerf" 또는 "3dovs"
CONFIG_FILE = f"configs/{DATASET}.yml"
SCENES = ["figurines"]                               # 실행할 장면
# SCENES = ["figurines", "ramen", "teatime", "waldo_kitchen"]  # LERF 전체
# SCENES = ["bed", "bench", "lawn", "room", "sofa"]           # 3DOVS 전체

print(f"Dataset: {DATASET}")
print(f"Config: {CONFIG_FILE}")
print(f"Scenes: {SCENES}")

In [ ]:
# 데이터셋 다운로드 (Google Drive → gdown)
import os
import gdown
import zipfile
import glob

# ---- Google Drive IDs ----
LERF_DATA_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"          # LERF-OVS zip
LERF_DATA_ZIP = "/content/lerf_ovs.zip"
DOVS3_FOLDER_ID = "1kdV14Gu5nZX6WOPbccG7t7obP_aXkOuC"        # 3DOVS folder
PRETRAINED_FOLDER_ID = "1b3bXy8XENhvpWh4nLzu06UPBZEZNX6fG"    # Pre-trained scenes folder

def extract_all_zips(src_dir, dst_dir):
    """src_dir 내 모든 zip을 dst_dir로 해제 후 삭제"""
    os.makedirs(dst_dir, exist_ok=True)
    for zf in glob.glob(f"{src_dir}/*.zip"):
        name = os.path.splitext(os.path.basename(zf))[0]
        print(f"  {name}.zip 압축 해제 중...")
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(dst_dir)
        os.remove(zf)

# ============================================================
# LERF 데이터셋
# ============================================================
if DATASET == "lerf":
    DATA_DIR = "data/lerf-ovs"
    OUTPUT_DIR = "output/lerf"

    # [1/2] LERF-OVS 데이터셋
    if not os.path.exists(f"{DATA_DIR}/figurines"):
        print("[1/2] LERF-OVS 데이터셋 다운로드 중...")
        gdown.download(id=LERF_DATA_ID, output=LERF_DATA_ZIP, quiet=False)
        print("압축 해제 중...")
        with zipfile.ZipFile(LERF_DATA_ZIP, 'r') as z:
            z.extractall("data/")
        os.remove(LERF_DATA_ZIP)
        # zip이 lerf_ovs(언더스코어)로 풀림 → config는 lerf-ovs(하이픈) 기대
        if not os.path.exists(DATA_DIR) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_DIR)
            print(f"심볼릭 링크: {DATA_DIR} -> data/lerf_ovs")
        print(f"LERF-OVS 완료: {os.listdir(DATA_DIR)}")
    else:
        print(f"LERF-OVS 이미 존재: {os.listdir(DATA_DIR)}")

    # [2/2] Pre-trained 장면
    if not os.path.exists(f"{OUTPUT_DIR}/figurines"):
        print("\n[2/2] Pre-trained 2DGS 장면 다운로드 중...")
        # gdown은 Drive 폴더명 그대로 사용 (예: output/LERF-pretrained-thgs/)
        gdown.download_folder(id=PRETRAINED_FOLDER_ID, output="output/_pretrained_tmp", quiet=False)
        # 다운된 폴더 내 zip들을 output/lerf/로 해제
        tmp_dirs = glob.glob("output/_pretrained_tmp*")
        for tmp_dir in tmp_dirs:
            extract_all_zips(tmp_dir, OUTPUT_DIR)
            # tmp 폴더 정리
            import shutil
            shutil.rmtree(tmp_dir, ignore_errors=True)
        print(f"Pre-trained 장면 완료: {os.listdir(OUTPUT_DIR)}")
    else:
        print(f"Pre-trained 장면 이미 존재: {os.listdir(OUTPUT_DIR)}")

# ============================================================
# 3DOVS 데이터셋
# ============================================================
elif DATASET == "3dovs":
    DATA_DIR = "data/3DOVS"
    OUTPUT_DIR = "output/3dovs"

    if not os.path.exists(DATA_DIR) or len(os.listdir(DATA_DIR)) == 0:
        print("[1/2] 3DOVS 데이터셋 다운로드 중...")
        os.makedirs(DATA_DIR, exist_ok=True)
        gdown.download_folder(id=DOVS3_FOLDER_ID, output=DATA_DIR, quiet=False)
        print("3DOVS 완료")
    else:
        print(f"3DOVS 이미 존재: {os.listdir(DATA_DIR)}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if len(os.listdir(OUTPUT_DIR)) == 0:
        print("\n[2/2] Pre-trained 장면 다운로드 중...")
        gdown.download_folder(id=PRETRAINED_FOLDER_ID, output="output/_pretrained_tmp", quiet=False)
        tmp_dirs = glob.glob("output/_pretrained_tmp*")
        for tmp_dir in tmp_dirs:
            extract_all_zips(tmp_dir, OUTPUT_DIR)
            import shutil
            shutil.rmtree(tmp_dir, ignore_errors=True)
        print(f"Pre-trained 장면 완료: {os.listdir(OUTPUT_DIR)}")
    else:
        print(f"Pre-trained 장면 이미 존재: {os.listdir(OUTPUT_DIR)}")

In [ ]:
# 다운로드 후 디렉토리 구조 확인
# 올바른 구조:
#   data/lerf-ovs/<scene>/images/, sparse/, (language_features/)
#   output/lerf/<scene>/point_cloud/iteration_30000/point_cloud.ply

print("=" * 60)
print("데이터 구조 확인")
print("=" * 60)

from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

for scene in SCENES:
    print(f"\n--- {scene} ---")
    data_path = f"{DATA_PATH}/{scene}"
    model_path = f"output/{SAVE_FOLDER}/{scene}"

    # 데이터 확인
    if os.path.exists(data_path):
        contents = os.listdir(data_path)
        has_images = "images" in contents
        has_sparse = "sparse" in contents
        print(f"  data ({data_path}): {contents}")
        if not has_images: print("    [!] images/ 폴더 없음")
        if not has_sparse: print("    [!] sparse/ 폴더 없음")
    else:
        print(f"  data: NOT FOUND at {data_path}")

    # 모델 확인
    ply_path = f"{model_path}/point_cloud/iteration_30000/point_cloud.ply"
    if os.path.exists(ply_path):
        size_mb = os.path.getsize(ply_path) / 1024 / 1024
        print(f"  model: point_cloud.ply ({size_mb:.1f} MB)")
    else:
        print(f"  model: NOT FOUND at {ply_path}")

---
## Cell 2. 언어 피처 생성 (SAM + CLIP)

각 이미지에서 SAM으로 multi-scale 마스크를 추출하고, CLIP으로 인코딩합니다.

> LangSplat 피처가 이미 있으면 (`language_features/` 폴더) 이 단계를 스킵합니다.

In [ ]:
# 언어 피처 존재 여부 확인
from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path

skip_encoding = True
for scene in SCENES:
    feat_path = f"{DATA_PATH}/{scene}/language_features"
    if os.path.exists(feat_path) and len(os.listdir(feat_path)) > 0:
        print(f"  {scene}: language_features 존재 ({len(os.listdir(feat_path))} files) - 스킵 가능")
    else:
        print(f"  {scene}: language_features 없음 - 생성 필요")
        skip_encoding = False

if skip_encoding:
    print("\n모든 장면에 language_features 존재. 다음 Cell로 넘어가세요.")

In [ ]:
# SAM 체크포인트 다운로드 (~2.5GB)
SAM_CKPT = "ckpts/sam_vit_h_4b8939.pth"
if not skip_encoding and not os.path.exists(SAM_CKPT):
    os.makedirs("ckpts", exist_ok=True)
    !wget -q --show-progress -O {SAM_CKPT} \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print("SAM 체크포인트 다운로드 완료")
else:
    print("SAM 체크포인트 불필요 또는 이미 존재")

In [ ]:
# 언어 피처 생성
if not skip_encoding:
    for scene in SCENES:
        feat_path = f"{DATA_PATH}/{scene}/language_features"
        if os.path.exists(feat_path) and len(os.listdir(feat_path)) > 0:
            print(f"{scene}: 이미 존재, 스킵")
            continue
        print(f"\n{'='*50}")
        print(f"{scene}: 언어 피처 생성 시작")
        print(f"{'='*50}")
        !python scripts/image_encoding.py \
            --source_path {DATA_PATH}/{scene} \
            --sam_ckpt_path {SAM_CKPT}
else:
    print("스킵: 모든 장면에 language_features가 이미 있습니다.")

---
## Cell 3. THGS 파이프라인 실행

핵심 4단계를 순서대로 실행합니다:
1. **인접 그래프 구축** - FRNN K-NN → `neighbor.pt`
2. **SAM 기반 엣지 가중치 조정** → `neighbor_new.pt`
3. **슈퍼포인트 분할** - Parallel Cut Pursuit → `nag-l1.pt`
4. **계층적 병합 + 피처 재투영** → `sai_nag.pt`

In [ ]:
# 전체 파이프라인 실행 (run.sh)
scenes_arg = " ".join(SCENES)
print(f"실행: bash scripts/run.sh {CONFIG_FILE} {scenes_arg}")
print("=" * 60)
!bash scripts/run.sh {CONFIG_FILE} {scenes_arg}

In [ ]:
# (선택) 개별 단계 실행 - 디버깅 시 사용
# 위 셀을 실행했으면 이 셀은 스킵

RUN_INDIVIDUAL = False

if RUN_INDIVIDUAL:
    print("[Step 3-1] 인접 그래프 구축 (FRNN K-NN)")
    print("=" * 60)
    !python scripts/launcher.py -f sp_partition.py -cf {CONFIG_FILE} -sc {scenes_arg}

    print("\n[Step 3-2] SAM 기반 엣지 가중치 조정")
    print("=" * 60)
    !python scripts/launcher.py -f graph_weight.py -cf {CONFIG_FILE} -sc {scenes_arg}

    print("\n[Step 3-3] 슈퍼포인트 분할 (Parallel Cut Pursuit)")
    print("=" * 60)
    !python scripts/launcher.py -f sp_partition.py -cf {CONFIG_FILE} -sc {scenes_arg} -k

    print("\n[Step 3-4] 계층적 병합 + CLIP 피처 재투영")
    print("=" * 60)
    !python scripts/launcher.py -f merge_proj.py -cf {CONFIG_FILE} -sc {scenes_arg}

In [ ]:
# 파이프라인 산출물 확인
from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
SAVE_FOLDER = cfg.dataset.save_folder

print("파이프라인 산출물 확인:")
print("=" * 60)
for scene in SCENES:
    model_path = f"output/{SAVE_FOLDER}/{scene}"
    print(f"\n--- {scene} ---")

    files_to_check = [
        ("neighbor.pt",     "Step 3-1: K-NN 인접 그래프"),
        ("neighbor_new.pt", "Step 3-2: 재가중된 그래프"),
        ("nag-l1.pt",       "Step 3-3: 슈퍼포인트 분할"),
        ("sai_nag.pt",      "Step 3-4: 최종 계층적 그래프 + 피처"),
    ]
    for fname, desc in files_to_check:
        fpath = os.path.join(model_path, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f"  [OK] {fname} ({size_mb:.1f} MB) - {desc}")
        else:
            print(f"  [X]  {fname} - {desc}")

---
## Cell 4. 평가 (Open-vocabulary Segmentation)

- `test_lerf.py`: 텍스트 쿼리로 2D 마스크 예측
- `eval_seg.py`: 예측 마스크 vs GT 비교 (IoU 등)

In [ ]:
from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

PRED_PATH = f"output/render/{SAVE_FOLDER}"
GT_PATH = f"{DATA_PATH}/label"
os.makedirs(PRED_PATH, exist_ok=True)

# 각 장면에 대해 마스크 예측
for scene in SCENES:
    print(f"\n{'='*50}")
    print(f"{scene}: 마스크 예측 중...")
    print(f"{'='*50}")
    !python test_lerf.py \
        -s {DATA_PATH}/{scene} \
        -m output/{SAVE_FOLDER}/{scene} \
        --path_pred {PRED_PATH}

In [ ]:
# 정량 평가
scenes_list = " ".join(SCENES)
print("정량 평가:")
print("=" * 60)
!python scripts/eval_seg.py \
    --dataset {DATASET} \
    --scene_list {scenes_list} \
    --path_pred {PRED_PATH} \
    --path_gt {GT_PATH}

---
## Cell 5. 시각화

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def visualize_predictions(pred_path, scene_name, max_prompts=6):
    """예측 마스크와 GT를 나란히 시각화"""
    scene_dir = Path(pred_path) / scene_name
    if not scene_dir.exists():
        print(f"{scene_name}: 예측 결과 없음")
        return

    image_dirs = sorted([d for d in scene_dir.iterdir() if d.is_dir()])

    for img_dir in image_dirs:
        pred_files = sorted([f for f in img_dir.glob("*.png") if "_gt" not in f.name])
        gt_files = sorted(img_dir.glob("*_gt.png"))

        n = min(len(pred_files), max_prompts)
        if n == 0:
            continue

        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        if n == 1:
            axes = axes.reshape(2, 1)

        fig.suptitle(f"{scene_name} / {img_dir.name}", fontsize=14)

        for i in range(n):
            pred = cv2.imread(str(pred_files[i]), cv2.IMREAD_GRAYSCALE)
            prompt_name = pred_files[i].stem

            axes[0, i].imshow(pred, cmap="gray")
            axes[0, i].set_title(f"Pred: {prompt_name}", fontsize=10)
            axes[0, i].axis("off")

            gt_file = img_dir / f"{prompt_name}_gt.png"
            if gt_file.exists():
                gt = cv2.imread(str(gt_file), cv2.IMREAD_GRAYSCALE)
                axes[1, i].imshow(gt, cmap="gray")
                axes[1, i].set_title("GT", fontsize=10)
            axes[1, i].axis("off")

        plt.tight_layout()
        plt.show()

for scene in SCENES:
    visualize_predictions(PRED_PATH, scene)

In [ ]:
# (선택) 커스텀 텍스트 쿼리
import torch
import sys
sys.path.insert(0, ".")

from utils.vlm_utils import ClipSimMeasure
from nag_data import SemanticNAG
from omegaconf import OmegaConf

cfg = OmegaConf.load(CONFIG_FILE)
SAVE_FOLDER = cfg.dataset.save_folder

QUERY_SCENE = SCENES[0]
QUERY_TEXT = "cup"        # <-- 원하는 텍스트로 변경

nag_path = f"output/{SAVE_FOLDER}/{QUERY_SCENE}/sai_nag.pt"
if os.path.exists(nag_path):
    nag_data = torch.load(nag_path)
    snag = SemanticNAG(nag_data["nag"], nag_data["nag_feat"])

    vlm = ClipSimMeasure()
    vlm.load_model()
    vlm.encode_text(QUERY_TEXT)

    sims = [vlm.compute_similarity(f) for f in snag.feat]
    related = snag.get_related_gaussian(sims, topk=3, level=[2, 3])
    n_selected = (related > 0).sum().item()
    total = related.shape[0]

    print(f"Query: '{QUERY_TEXT}'")
    print(f"관련 가우시안: {n_selected} / {total} ({100 * n_selected / total:.1f}%)")
else:
    print(f"{nag_path} 없음 - Cell 3 파이프라인을 먼저 실행하세요")